In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/DataCoSupplyChain_Cleaned.csv")

df["order date (DateOrders)"] = pd.to_datetime(
    df["order date (DateOrders)"],
    errors="coerce"
)

df["shipping date (DateOrders)"] = pd.to_datetime(
    df["shipping date (DateOrders)"],
    errors="coerce"
)

print(df.shape)

(180519, 45)


In [7]:
# ============================================
# 1. CREATE LEAKAGE-SAFE ML FEATURE DATASET
# ============================================

target = "Late_delivery_risk"

# Create date-based features
df["Order_Year"] = df["order date (DateOrders)"].dt.year
df["Order_Month"] = df["order date (DateOrders)"].dt.month
df["Order_DayOfWeek"] = df["order date (DateOrders)"].dt.dayofweek

# Features available before or at order placement
features = [
    "Type",
    "Days for shipment (scheduled)",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Quantity",
    "Order Item Product Price",
    "Product Price",
    "Customer Segment",
    "Market",
    "Order Region",
    "Order Country",
    "Department Name",
    "Category Name",
    "Shipping Mode",
    "Order_Year",
    "Order_Month",
    "Order_DayOfWeek"
]

# Create ML dataset
ml_df = df[features + [target]].copy()

print("ML Dataset Shape:", ml_df.shape)

print("\nTarget Distribution:")
print(ml_df[target].value_counts())

print("\nMissing Values:")
print(ml_df.isnull().sum().sort_values(ascending=False).head(10))

ML Dataset Shape: (180519, 18)

Target Distribution:
Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Missing Values:
Type                             0
Days for shipment (scheduled)    0
Order Item Discount              0
Order Item Discount Rate         0
Order Item Quantity              0
Order Item Product Price         0
Product Price                    0
Customer Segment                 0
Market                           0
Order Region                     0
dtype: int64


In [8]:
# ============================================
# 2. ENCODE CATEGORICAL FEATURES
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

X = ml_df.drop(columns=[target])
y = ml_df[target]

categorical_features = X.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object", "string"]
).columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

X_encoded = preprocessor.fit_transform(X)

print("\nOriginal Feature Count:", X.shape[1])
print("Encoded Feature Count:", X_encoded.shape[1])
print("Encoded Dataset Shape:", X_encoded.shape)

Categorical Features:
['Type', 'Customer Segment', 'Market', 'Order Region', 'Order Country', 'Department Name', 'Category Name', 'Shipping Mode']

Numerical Features:
['Days for shipment (scheduled)', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Quantity', 'Order Item Product Price', 'Product Price', 'Order_Year', 'Order_Month', 'Order_DayOfWeek']

Original Feature Count: 17
Encoded Feature Count: 273
Encoded Dataset Shape: (180519, 273)


In [9]:
# ============================================
# 3. TRAIN / TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting Data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTarget Distribution:")
print("Training:")
print(y_train.value_counts(normalize=True))

print("\nTesting:")
print(y_test.value_counts(normalize=True))

Training Data:
X_train: (144415, 273)
y_train: (144415,)

Testing Data:
X_test: (36104, 273)
y_test: (36104,)

Target Distribution:
Training:
Late_delivery_risk
1    0.548288
0    0.451712
Name: proportion, dtype: float64

Testing:
Late_delivery_risk
1    0.548305
0    0.451695
Name: proportion, dtype: float64


In [10]:
# ============================================
# 4. SAVE LEAKAGE-SAFE ML PREPARATION
# ============================================

import joblib

joblib.dump(
    preprocessor,
    "../data/ml_preprocessor.pkl"
)

joblib.dump(
    X_train,
    "../data/X_train.pkl"
)

joblib.dump(
    X_test,
    "../data/X_test.pkl"
)

joblib.dump(
    y_train,
    "../data/y_train.pkl"
)

joblib.dump(
    y_test,
    "../data/y_test.pkl"
)

print("Leakage-safe ML preparation saved successfully.")

Leakage-safe ML preparation saved successfully.
